## 📑 Core Payment Jump
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQBtHQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
%python
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
%python
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

import pandas as pd
from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
%python
import boto3
import os

#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d') 

som_date = spark.sql(
            f"""
            SELECT MAX(CalendarDate) 
            FROM `{env_prefix}-cdh`.reference.ref_calendar 
            WHERE LastDateOfMonth =  last_day(add_months({run_date!r},-1))
            AND IsBusinessDay = 1
            """
        ).collect()[0][0]

In [0]:
table_name = "core_PaymentJump"

schema = (
         "BusinessDate DATE,"
         "ExposureKey STRING,"
         "Assignment STRING,"
         "AssetSegment STRING,"
         "Balance DECIMAL(19,2),"
         "EBAStatus STRING,"
         "Active STRING,"
         "InteractionTypeKey STRING,"
         "InteractionDate DATE,"
         "ActivityCode STRING,"
         "Subcategory STRING,"
         "Category STRING,"
         "Modification STRING,"
         "ProsarmozoDuration STRING,"
         "ModificationDate  DATE,"
         "Month  STRING,"
         "InLastAssignmentAndInOldPool  STRING,"
         "EBARisk  STRING,"
         "BucketRiskWithMateriality INT,"
         "CaseBucket INT,"
         "RiskFound STRING,"
         "OpsFound STRING,"
         "EbaRiskGroup STRING,"
         "BucketRiskGroup STRING,"
         "BucketOperationalGroup STRING,"
         "InsertTimeStamp timestamp"
         )

In [0]:
%python
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


In [0]:
%python
assignmentpoolrestcampaigns = spark.sql(
    f'''
SELECT    ExposureKey
        , NewAssignment
        , NewActive
        , Assignment
        , AssetSegment
        , cast(replace(regexp_replace(Balance, '[^0-9,]', ''),',', '.') AS DECIMAL(19,2)) AS Balance
        , EBAStatus
        , Active
        , ActivityTypeKey
        , ActivityDate
        , ActivityCode
        , Subcategory
        , Category
        , Modifications
        , ProsarmozoDuration
        , RestructuringDate
        , Month
        , InLastAssignmentAndInOldPool
        , Channel
        , AssignmentMonth
        , Flag
FROM `{env_prefix}-cdh`.externalfiles.extassignmentpoolrestcampaigns A
INNER JOIN (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp, BusinessDate FROM `{env_prefix}-cdh`.externalfiles.extassignmentpoolrestcampaigns GROUP BY BusinessDate) B
ON A.BusinessDate = B.BusinessDate AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Flag = 'Extensions'
 AND A.BusinessDate = (SELECT MAX(BusinessDate) FROM `{env_prefix}-cdh`.externalfiles.extassignmentpoolrestcampaigns  WHERE {run_date!r} >= BusinessDate)  
''')
assignmentpoolrestcampaigns.createOrReplaceTempView("assignmentpoolrestcampaigns")  

# Check if the dataset is empty and print a message accordingly
if assignmentpoolrestcampaigns.count() == 0:
    print(f"The required external file extassignmentpoolrestcampaigns dataset for dates <= {run_date!r} is empty")

casecode = spark.sql(
f'''
select distinct d.Portfolioname,  d.exposurekey ,d.casecode
from `{env_prefix}-cdh`.primary.casecode c
inner join `{env_prefix}-cdh`.primary.exposure d on c.casecodekey = d.casecode and c.PortfolioName = d.Portfolioname
where d.Portfolioname = 'Core Retail'
'''
)
casecode.createOrReplaceTempView("casecode")

interaction = spark.sql(
f'''
select
       l.ExposureKey
     , l.PortfolioName
     , l.InteractionDate
     , l.InteractionTypeKey
     , l.InteractionType
     , l.CDI
 FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
 inner join assignmentpoolrestcampaigns a on l.ExposureKey = a.ExposureKey
where 1=1
and cast(CreatedDate as date) >= '2022-07-01'  AND cast(CreatedDate as date) <= {run_date!r}
and l.SourceSystem='qca'
and l.PortfolioName = 'Core Retail'
and InteractionTypeKey IN ('152','545','620','621','622','623','624')
 
union
 
select
      c.ExposureKey
     , l.PortfolioName
     , l.InteractionDate
     , l.InteractionTypeKey
     , l.InteractionType
     , l.CDI
 FROM `{env_prefix}-cdh`.primary.get_interactions_by_business_date({run_date!r}) l
 inner join casecode c on c.casecode = l.casecode
 inner join assignmentpoolrestcampaigns a on c.ExposureKey = a.ExposureKey
where 1=1
and cast(CreatedDate as date) >= '2022-07-01'  AND cast(CreatedDate as date) <= {run_date!r}
and l.SourceSystem='qca'
and l.PortfolioName = 'Core Retail'
and InteractionTypeKey IN ('152','545','620','621','622','623','624')
''')
interaction.createOrReplaceTempView("interaction")


dexposure= spark.sql(
f'''
select
     ExposureKey
    ,CDI
    ,Bucket
from `{env_prefix}-cdh`.primary.exposure
where BusinessDate = {run_date!r}
and PortfolioName = 'Core Retail'
and PartyRelationShipType = 'Primary Owner'
and IsActiveFlag = true
'''
)
dexposure.createOrReplaceTempView("dexposure")

unionfile= spark.sql(
f'''
select 
     uf.ExposureNumber
    ,uf.BucketCategoryWithMateriality  
    ,uf.SnapshotDate
    ,uf.EBAStatus
from `{env_prefix}-cdh`.externalfiles.extunionfile uf
join (select SnapshotDate, max(IngestionTimestamp) IngestionTimestamp from `{env_prefix}-cdh`.externalfiles.extunionfile group by SnapshotDate) uf0
on uf.IngestionTimestamp = uf0.IngestionTimestamp and uf.SnapshotDate = uf0.SnapshotDate
where uf.SnapshotDate = (select max(SnapshotDate) from `{env_prefix}-cdh`.externalfiles.extunionfile where SnapshotDate < {run_date!r})

'''
)
unionfile.createOrReplaceTempView("unionfile")

##### Activities

In [0]:
 %python
Activities = spark.sql(
f'''
 SELECT
              a.ExposureKey
            , a.InteractionDate
            , a.InteractionTypeKey
			      , CASE WHEN ac.ExposureKey = a.ExposureKey AND ac.CDI = a.CDI THEN 1 ELSE 0 END AS PrimaryDebtorFlag
         	  , RANK() OVER(PARTITION BY a.ExposureKey ORDER BY a.InteractionDate DESC, CASE WHEN ac.ExposureKey = a.ExposureKey AND ac.CDI = a.CDI THEN 1 ELSE 0 END DESC, a.InteractionTypeKey DESC) AS Rank
            , am.ActivityCode
            , am.Subcategory
            , am.category
    FROM interaction          a
	  LEFT JOIN dexposure ac
		ON ac.ExposureKey = a.ExposureKey --AND ac.CDI = a.CDI
    LEFT JOIN `{env_prefix}-reportinputfiles`.repoint.core_PaymentJumpActivityMapping am
    ON a.InteractionTypeKey = am.Code   
    '''
)
Activities.createOrReplaceTempView("Activities")

##### Extension

In [0]:
Extension_1 = spark.sql(
f'''    
SELECT    Month
        , ModificationDate
        , OldExposureKey
        , Categoryfinal 
FROM (
SELECT    Month
        , ModificationDate
        , OldExposureKey
        , Categoryfinal 
        , 1 AS Ranking
FROM `{env_prefix}-reportinputfiles`.repoint.core_statichistoricalextensionbeforemigration
	        
UNION 
	        
SELECT  
	      Month
            , ModificationDate
            , OldExposureKey
            , Categoryfinal
	    , RANK() OVER (PARTITION BY ModificationDate, OldExposureKey, NewExposureKey  ORDER BY BusinessDate DESC) AS Ranking
FROM `{env_prefix}-cdh-delivery`.repoint.core_modifications 
WHERE Categoryfinal = 'Extension' 
AND ModificationDate > (select max(ModificationDate) from `{env_prefix}-reportinputfiles`.repoint.core_statichistoricalextensionbeforemigration) 
AND ModificationDate <={run_date!r}
) AS a
WHERE Ranking = 1
''')
Extension_1.createOrReplaceTempView("Extension_1")    

In [0]:
Extension = spark.sql(
f'''    
SELECT    Month
        , ModificationDate
        , OldExposureKey
        , Categoryfinal 
FROM (
SELECT  
          Month
        , ModificationDate
        , OldExposureKey
        , Categoryfinal 
		, RANK() OVER (PARTITION BY OldExposureKey  ORDER BY ModificationDate DESC) AS Ranking
FROM Extension_1 
	) AS a
WHERE Ranking = 1
''')
Extension.createOrReplaceTempView("Extension") 

##### Fast Track

In [0]:
FastTrack = spark.sql(
f'''    
		SELECT DISTINCT ExposureKey
                      , ProsarmozoDuration
        FROM (
        SELECT ExposureKey 
             , CASE WHEN ProsarmozoDuration = '12' THEN '012' 
        	        WHEN ProsarmozoDuration = '24' THEN '024'
        			ELSE ProsarmozoDuration 
               END AS ProsarmozoDuration
        FROM `{env_prefix}-cdh`.externalfiles.extfasttrack
		WHERE BusinessDate >= '2023-10-01'
        
        UNION ALL
        
        SELECT ExposureKey
             , ProsarmozoDuration
        FROM `{env_prefix}-reportinputfiles`.repoint.core_statichistoricalfasttrackbeforemigration
		WHERE Flag = 'PaymentJump'
              ) AS a
''')
FastTrack.createOrReplaceTempView("FastTrack")                

##### StepUp

In [0]:
StepupImplementationsStep0 = spark.sql(
f'''    
SELECT DISTINCT ExposureKey
              , ProsarmozoDuration
FROM `{env_prefix}-cdh`.externalfiles.extextensionstepupimplementations a
INNER JOIN (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp, BusinessDate FROM `{env_prefix}-cdh`.externalfiles.extextensionstepupimplementations GROUP BY BusinessDate) b
ON a.BusinessDate = b.BusinessDate and a.IngestionTimeStamp = b.IngestionTimeStamp
WHERE a.BusinessDate = (SELECT MAX(BusinessDate) FROM `{env_prefix}-cdh`.externalfiles.extextensionstepupimplementations  WHERE {run_date!r} >= BusinessDate)  

''')
StepupImplementationsStep0.createOrReplaceTempView("StepupImplementationsStep0")

# Check if the dataset is empty and print a message accordingly
if StepupImplementationsStep0.count() == 0:
    print(f"The required external file extextensionstepupimplementations dataset for dates <= {run_date!r} is empty")

##### Assignment

In [0]:
Assignment_V1 = spark.sql(
f''' 
        SELECT 
               a.ExposureKey
             , a.Assignment                                                           
             , a.AssetSegment 
             , a.Balance                                                             
             , a.EBAStatus                                                            
             , a.Active
             , act.InteractionTypeKey
             , act.InteractionDate
             , act.ActivityCode
             , CASE WHEN a.ExposureKey = e.OldExposureKey THEN 'Positive With Application' ELSE act.Subcategory END AS Subcategory
             , CASE WHEN a.ExposureKey = e.OldExposureKey THEN 'Positive' ELSE act.Category END AS Category
             , CASE WHEN a.ExposureKey = e.OldExposureKey THEN 'Yes' END AS Modification
             , CASE WHEN a.ExposureKey = s.ExposureKey THEN s.ProsarmozoDuration ELSE f.ProsarmozoDuration END AS ProsarmozoDuration
             , CASE WHEN a.ExposureKey = e.OldExposureKey THEN e.ModificationDate END AS ModificationDate
             , CASE WHEN a.ExposureKey = e.OldExposureKey THEN e.Month END AS Month
             , a.InLastAssignmentAndInOldPool
        FROM assignmentpoolrestcampaigns     a
        LEFT JOIN Activities      act
        ON a.ExposureKey = act.ExposureKey and act.Rank = 1
        LEFT JOIN FastTrack f
        ON a.ExposureKey = f.ExposureKey
        LEFT JOIN Extension e
        ON a.ExposureKey = e.OldExposureKey
        LEFT JOIN StepupImplementationsStep0     s
        ON a.ExposureKey = s.ExposureKey
''')
Assignment_V1.createOrReplaceTempView("Assignment_V1")        

In [0]:
Assignment_V2 = spark.sql(
f''' 
        SELECT DISTINCT a1.ExposureKey
             , a1.Assignment 
             , a1.AssetSegment 
             , a1.Balance
             , a1.EBAStatus 
             , a1.Active
             , a1.InteractionTypeKey
             , a1.InteractionDate
             , a1.ActivityCode
             , a1.Subcategory
             , a1.Category
             , a1.Modification
             , a1.ProsarmozoDuration
             , a1.ModificationDate
             , a1.Month
             , a1.InLastAssignmentAndInOldPool
             , CASE WHEN a1.Modification = 'Yes' AND u.ExposureNumber = a1.ExposureKey THEN u.EBAStatus END AS EbaRisk
             , CASE WHEN a1.Modification = 'Yes' AND u.ExposureNumber = a1.ExposureKey THEN u.BucketCategoryWithMateriality END AS BucketRiskWithMateriality
             , CASE WHEN a1.Modification = 'Yes' AND u.ExposureNumber = a1.ExposureKey THEN 'YES' ELSE 'n/a' END AS RiskFound
             , CASE WHEN a1.Modification = 'Yes' AND de.ExposureKey = a1.ExposureKey  THEN de.Bucket
		            WHEN a1.Modification = 'Yes' AND a1.ExposureKey = q.ACCOUNT_NUMBER THEN q.CURRENT_BUCKET
	           END AS CaseBucket
             , CASE WHEN a1.Modification = 'Yes' AND a1.ExposureKey = q.ACCOUNT_NUMBER THEN 'YES'
		           WHEN a1.Modification = 'Yes' AND de.ExposureKey = a1.ExposureKey THEN 'YES' 
                   ELSE 'n/a'
               END AS OpsFound
        FROM Assignment_V1                      a1
        LEFT JOIN Unionfile u ON u.ExposureNumber = a1.ExposureKey
        LEFT JOIN dexposure de ON de.ExposureKey = a1.ExposureKey
        LEFT JOIN `{env_prefix}-cdh-delivery`.repoint.core_retailprimary q ON a1.ExposureKey = q.ACCOUNT_NUMBER AND q.BusinessDate = {som_date!r}
''')
Assignment_V2.createOrReplaceTempView("Assignment_V2")        

##### PaymentJump

In [0]:
PaymentJump = spark.sql(
f''' 
        SELECT ExposureKey
             , Assignment 
             , AssetSegment 
             , cast(replace(Balance,',','.') as decimal(19,2)) AS Balance
             , EBAStatus 
             , Active
             , InteractionTypeKey
             , InteractionDate
             , ActivityCode
             , Subcategory
             , Category
             , Modification
             , ProsarmozoDuration
             , ModificationDate
             , Month
             , InLastAssignmentAndInOldPool
             , EbaRisk
             , BucketRiskWithMateriality
             , RiskFound
             , CaseBucket
             , OpsFound
             , CASE WHEN EbaRisk='FNPL' OR EbaRisk='NPL' THEN 'NPE'
                    WHEN EbaRisk='PL' THEN 'PL'
             	    WHEN EbaRisk='FPL' THEN 'FPL'
                    END AS EbaRiskGroup
             , CASE WHEN BucketRiskWithMateriality>3 THEN '4+' 
                    WHEN BucketRiskWithMateriality='3' THEN '3'
             	    WHEN BucketRiskWithMateriality='2' THEN '2'
             	    WHEN BucketRiskWithMateriality='1' THEN '1'
             	    WHEN BucketRiskWithMateriality='0' THEN '0'
                    END AS BucketRiskGroup
             , CASE WHEN CaseBucket>'3' OR CaseBucket>'03' THEN '4+'
                    WHEN CaseBucket='3' OR CaseBucket='03' THEN '3'
             	    WHEN CaseBucket='2' OR CaseBucket='02' THEN '2'
             	    WHEN CaseBucket='1' OR CaseBucket='04' THEN '1'
             	    WHEN CaseBucket='0' OR CaseBucket='00' THEN '0'
                    END AS BucketOperationalGroup
        FROM Assignment_V2                      a1
''')
PaymentJump.createOrReplaceTempView("PaymentJump")        

##### final_df

In [0]:
final_df = spark.sql(
f'''
		SELECT      CAST({run_date!r} AS DATE)							  BusinessDate
		    ,       CAST(ExposureKey AS STRING)                   		  ExposureKey
            ,       CAST(Assignment AS STRING)              			  Assignment
            ,       CAST(AssetSegment AS STRING)                          AssetSegment
            ,       CAST(Balance AS DECIMAL(19,2))						  Balance
            ,       CAST(EBAStatus AS STRING)                   		  EBAStatus
            ,       CAST(Active AS STRING)								  Active
            ,       CAST(InteractionTypeKey AS STRING)                    InteractionTypeKey
            ,       CAST(InteractionDate AS DATE)                  		  InteractionDate
            ,       CAST(ActivityCode AS STRING)                		  ActivityCode
            ,       CAST(Subcategory AS STRING)							  Subcategory
            ,       CAST(Category AS STRING)							  Category
            ,       CAST(Modification AS STRING)						  Modification
            ,       CAST(ProsarmozoDuration AS STRING)          		  ProsarmozoDuration
            ,       CAST(ModificationDate AS DATE)          		      ModificationDate
            ,       CAST(Month AS STRING)								  Month
            ,       CAST(InLastAssignmentAndInOldPool AS STRING)		  InLastAssignmentAndInOldPool
            ,       CAST(EBARisk AS STRING)                     		  EBARisk
            ,       CAST(BucketRiskWithMateriality AS INT) 		          BucketRiskWithMateriality
            ,       CAST(CaseBucket AS int)                  		      CaseBucket
            ,       CAST(RiskFound AS STRING)                   		  RiskFound
            ,       CAST(OpsFound AS STRING)                    		  OpsFound
            ,       CAST(EbaRiskGroup AS STRING)               		      EbaRiskGroup
            ,       CAST(BucketRiskGroup AS STRING)            		      BucketRiskGroup
            ,       CAST(BucketOperationalGroup AS STRING)     		      BucketOperationalGroup
			,       CAST(GETDATE() AS timestamp)                          InsertTimeStamp
		FROM PaymentJump
''')
final_df.createOrReplaceTempView("final_df")  

In [0]:
%python
if final_df.rdd.isEmpty():
    raise ValueError(f"Core_PaymentJump: final DataFrame is empty - aborting write to {delivery_table_path} to avoid overwriting with no data")
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="warning")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
PaymentJump = spark.sql(
f'''
select * from `{env_prefix}-cdh-delivery`.repoint.core_PaymentJump
where BusinessDate = {run_date!r}
'''
)
PaymentJump.createOrReplaceTempView("PaymentJump")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/Core/PaymentJump/"
file_name = "PaymentJump_" +run_date+ ".csv"
PaymentJump.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}")
rename_and_cleanup_files(output_path, file_name)